In [ ]:
### IMPORTS
import pandas as pd
import numpy as np
import os 
from backtester.pnl import pnl
from backtester.metrics import compute_metrics
from utils.io import load_partitioned_parquet
from utils.paths import make_data_path
print("DONE")

###CONFIG PLEASE
MAKER_FEE = 0.000200
TAKER_FEE = 0.000550
FUNDING_INTERVAL = 480 # 8 hours in minutes
SYMBOL = 'BTCUSDT'
INTERVAL = 60
START = '01/01/2026'
END = None

DONE


In [ ]:
### THis needs to be a function too
### LOAD DATA

cols = ['mark_close'] # Whatever I want for the strategy 
path_ohlcv = make_data_path('ohlcv', SYMBOL, INTERVAL)
df_ohlcv = load_partitioned_parquet(path_ohlcv, start=START, end=END, columns=cols)

if df_ohlcv.empty:
    raise ValueError("df_ohlcv loaded empty. Check path or date filters.")

path_funding = make_data_path('funding', SYMBOL)
df_funding = load_partitioned_parquet(path_funding,start=START, end=END)

if df_funding.empty:
    raise ValueError("df_funding loaded empty. Check path or date filters.")

df_merge = df_ohlcv.merge(df_funding, how='left', on='timestamp')
df_merge['fundingRate'] = df_merge['fundingRate'].fillna(0)
df_merge = df_merge.sort_index()

mark_close = df_merge["mark_close"].astype("float64").to_numpy()
funding  = df_merge["fundingRate"].astype("float64").to_numpy()


                           mark_close  last_open  fundingRate
timestamp                                                    
2026-01-01 00:00:00+00:00    87760.45    87594.9     0.000036
2026-01-01 01:00:00+00:00    87916.38    87758.5     0.000000
2026-01-01 02:00:00+00:00    87876.07    87914.0     0.000000
2026-01-01 03:00:00+00:00    87820.00    87875.9     0.000000
2026-01-01 04:00:00+00:00    87555.80    87822.0     0.000000


In [ ]:
pos = np.ones(len(mark_close)) # always long lol
print(len(pos))

1205


In [18]:
pnl_df = pnl(df_merge, pos)
print(pnl_df.head(10))
print(pnl_df.tail(10))

                           strategy_ret    equity  held_pos  trade     fees  \
timestamp                                                                     
2026-01-01 00:00:00+00:00      0.000000  1.000000       0.0    1.0  0.00055   
2026-01-01 01:00:00+00:00      0.001777  1.001777       1.0    0.0  0.00000   
2026-01-01 02:00:00+00:00     -0.000459  1.001317       1.0    0.0  0.00000   
2026-01-01 03:00:00+00:00     -0.000638  1.000679       1.0    0.0  0.00000   
2026-01-01 04:00:00+00:00     -0.003008  0.997668       1.0    0.0  0.00000   
2026-01-01 05:00:00+00:00      0.001283  0.998948       1.0    0.0  0.00000   
2026-01-01 06:00:00+00:00     -0.000590  0.998358       1.0    0.0  0.00000   
2026-01-01 07:00:00+00:00     -0.000341  0.998018       1.0    0.0  0.00000   
2026-01-01 08:00:00+00:00      0.001548  0.999563       1.0    0.0  0.00000   
2026-01-01 09:00:00+00:00      0.001227  1.000790       1.0    0.0  0.00000   

                           funding_pnl  
timestamp 